<div dir="rtl">

# الدرس 33: ضبط المعاملات الفائقة

دفتر الكود لهذا الدرس. شغّل الخلايا بالترتيب من الأعلى.

</div>

<div dir="rtl">

### البحث الشبكي داخل خط معالجة

</div>

In [ ]:
import numpy as np
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split, GridSearchCV, StratifiedKFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC

X, y = load_breast_cancer(return_X_y=True)
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.2, random_state=0, stratify=y)

pipe = Pipeline([("scale", StandardScaler()), ("svc", SVC())])
grid = {"svc__C": [0.1, 1, 10, 100], "svc__gamma": [0.001, 0.01, 0.1]}
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=0)

search = GridSearchCV(pipe, grid, cv=cv).fit(X_tr, y_tr)
print("best params:", search.best_params_)
print("best CV accuracy:", round(search.best_score_, 4))

<div dir="rtl">

### جدول كل التجارب

</div>

In [ ]:
import pandas as pd

res = pd.DataFrame(search.cv_results_)
table = res.pivot(index="param_svc__C", columns="param_svc__gamma", values="mean_test_score")
print(table.round(3))

<div dir="rtl">

### التقييم النهائي، مرة واحدة

</div>

In [ ]:
print("test accuracy of the refitted best model:", round(search.score(X_te, y_te), 4))

<div dir="rtl">

### البحث العشوائي للتعزيز

</div>

In [ ]:
from sklearn.model_selection import RandomizedSearchCV
from sklearn.ensemble import HistGradientBoostingClassifier
from scipy.stats import loguniform, randint

space = {
    "learning_rate": loguniform(0.01, 0.3),     # مقياس لوغاريتمي
    "max_depth": randint(2, 8),
    "max_leaf_nodes": randint(8, 64),
    "l2_regularization": loguniform(1e-4, 1),
}
rs = RandomizedSearchCV(HistGradientBoostingClassifier(random_state=0), space,
                        n_iter=20, cv=cv, random_state=0).fit(X_tr, y_tr)
best = {k: (round(float(v), 4) if isinstance(v, float) else v) for k, v in rs.best_params_.items()}
print("best params:", best)
print("best CV accuracy:", round(rs.best_score_, 4), " test accuracy:", round(rs.score(X_te, y_te), 4))

<div dir="rtl">

## البروتوكول الكامل

</div>

In [ ]:
from sklearn.model_selection import cross_val_score

inner = GridSearchCV(pipe, grid, cv=StratifiedKFold(3, shuffle=True, random_state=1))
outer_scores = cross_val_score(inner, X_tr, y_tr, cv=StratifiedKFold(5, shuffle=True, random_state=2))
print(f"nested CV: {outer_scores.mean():.3f} ± {outer_scores.std():.3f}")

<div dir="rtl">

## تمرين: ابحث عن أفضل غابة

استخدم نفس تقسيم بيانات أورام الثدي.

1. ابحث بالشبكة عن أفضل غابة عشوائية، بهذه القيم: عدد الأشجار 100 و 300، وأقصى عمق 3 و 6 وبلا حد، وأقل عدد أمثلة في الورقة 1 و 5.
2. كم تدريباً سيحدث مع خمس طيات؟ احسبه قبل أن تشغّل.
3. اطبع أفضل المعاملات، وأفضل رقم تحقق، ثم رقم الاختبار مرة واحدة.

</div>

In [ ]:
from sklearn.ensemble import RandomForestClassifier

grid_rf = {
    # اكتب القيم هنا
}